# Statewide Property Tax — Historic Series

**Source:** `PVDHistoricTax.pdf`, Kansas Department of Revenue, Property Valuation
Division (PVD) Statistical Report of Property Assessment and Taxation.

**Goal:** Parse the statewide historic total property tax collected (1998–2025),
broken out by property class (Residential, Commercial & Industrial, Utilities,
Ag Land, Oil & Gas, All Other), to complement the existing `statewide_valuation`
table and provide a real reported tax-dollar figure rather than a computed
estimate (assessed value × mill levy).

This feeds directly into the core project question: **is Kansas property tax
growth driven more by rising assessed valuations or rising mill levy rates?**
A reported total-tax series lets us check that story against the actual dollar
outcome, not just the two inputs.

In [7]:
import pdfplumber

with pdfplumber.open("../data/raw/kdor/PVDHistoricTax.pdf") as pdf:
    page = pdf.pages[0]
    raw_text = page.extract_text()

print(raw_text)

Tax Dollars
Major Classes of Property (Millions)
Year Residential % of Total C&I Real/PP % of Total Utilities % of Total Ag Land % of Total Oil & Gas % of Total All Other % of Total Tax
98 $798.961 40.59 $594.922 30.23 $267.176 13.57 $134.835 6.86 $103.552 5.26 $65.103 3.31 $1,964.549
99 $878.324 41.63 $653.373 30.97 $284.341 13.48 $144.150 6.83 $76.320 3.62 $69.079 3.28 $2,105.586
00 $982.067 42.53 $713.499 30.90 $289.787 12.55 $156.938 6.80 $83.015 3.60 $78.475 3.41 $2,303.781
01 $1,095.606 42.04 $770.894 30.26 $300.918 10.81 $171.704 5.78 $115.393 4.53 $87.927 3.46 $2,542.442
02 $1,175.185 44.23 $799.238 30.08 $299.439 11.27 $184.307 6.94 $105.025 3.95 $88.167 3.33 $2,651.361
03 $1,261.071 45.30 $831.869 29.89 $311.099 11.18 $183.373 6.59 $99.585 3.58 $91.211 3.28 $2,778.207
04 $1,355.269 45.64 $865.551 29.14 $329.988 11.08 $189.635 6.39 $131.039 4.41 $92.064 3.11 $2,963.545
05 $1,461.705 45.95 $918.794 28.88 $337.871 10.62 $188.601 5.93 $169.892 5.34 $98.187 3.09 $3,175.050
06 $1,5

## Extraction

Text-layer extraction with `pdfplumber` (Poppler's `pdftotext` isn't installed
on this machine, so `pdfplumber` was used instead — same result, pure Python,
no external binary required).

The source table lists years as 2-digit values (`98`–`25`); these are converted
to 4-digit years (1998–2025) during parsing. Dollar figures are reported in
millions.

In [8]:
import re
import pandas as pd

categories = ['residential', 'ci_real_pp', 'utilities', 'ag_land', 'oil_gas', 'all_other']

records = []

for line in raw_text.split("\n"):
    line = line.strip()

    # Only process lines that start with a 2-digit year followed by data
    match = re.match(r'^(\d{2})\s+(.*)$', line)
    if not match:
        continue

    year_2digit = match.group(1)
    rest = match.group(2)

    # Skip if this doesn't actually contain dollar signs (filters out junk lines)
    if '$' not in rest:
        continue

    # Split on '$' — first chunk is empty, then 6 "amount percent" pairs, then 1 total
    parts = [p.strip() for p in rest.split('$') if p.strip()]

    if len(parts) != 7:
        continue  # not a real data row, skip it

    # Convert 2-digit year to 4-digit year
    yy = int(year_2digit)
    year = 1900 + yy if yy >= 90 else 2000 + yy

    record = {'year': year}

    for i, cat in enumerate(categories):
        amt_str, pct_str = parts[i].split()
        record[f'{cat}_millions'] = float(amt_str.replace(',', ''))
        record[f'{cat}_pct'] = float(pct_str)

    record['total_tax_millions'] = float(parts[6].replace(',', ''))

    records.append(record)

historic_tax_df = pd.DataFrame(records)
print(f"Rows parsed: {len(historic_tax_df)}")
historic_tax_df

Rows parsed: 28


,year,residential_millions,residential_pct,ci_real_pp_millions,ci_real_pp_pct,utilities_millions,utilities_pct,ag_land_millions,ag_land_pct,oil_gas_millions,oil_gas_pct,all_other_millions,all_other_pct,total_tax_millions
0,1998,798.961,40.59,594.922,30.23,267.176,13.57,134.835,6.86,103.552,5.26,65.103,3.31,1964.549
1,1999,878.324,41.63,653.373,30.97,284.341,13.48,144.150,6.83,76.320,3.62,69.079,3.28,2105.586
2,2000,982.067,42.53,713.499,30.90,289.787,12.55,156.938,6.80,83.015,3.60,78.475,3.41,2303.781
3,2001,1095.606,42.04,770.894,30.26,300.918,10.81,171.704,5.78,115.393,4.53,87.927,3.46,2542.442
4,2002,1175.185,44.23,799.238,30.08,299.439,11.27,184.307,6.94,105.025,3.95,88.167,3.33,2651.361
5,2003,1261.071,45.30,831.869,29.89,311.099,11.18,183.373,6.59,99.585,3.58,91.211,3.28,2778.207
6,2004,1355.269,45.64,865.551,29.14,329.988,11.08,189.635,6.39,131.039,4.41,92.064,3.11,2963.545
7,2005,1461.705,45.95,918.794,28.88,337.871,10.62,188.601,5.93,169.892,5.34,98.187,3.09,3175.050
8,2006,1576.312,46.03,985.890,28.78,341.681,9.98,184.285,5.38,225.778,6.59,104.429,3.05,3418.375
9,2007,1691.947,46.91,1052.047,29.17,354.843,9.84,174.575,4.84,215.461,5.97,111.162,3.09,3600.036


## Validation

Parsed row count and two endpoint values were checked against the raw PDF text
directly:
- 28 rows (1998–2025), matching the existing `statewide_valuation` table's range
- 1998 total tax: $1,964.549M ✅ matches source
- 2025 total tax: $6,815.142M ✅ matches source

In [9]:
import sqlite3

conn = sqlite3.connect("../data/processed/kansas_tax.db")
historic_tax_df.to_sql("statewide_tax", conn, if_exists="replace", index=False)

# Verify it landed correctly
check = pd.read_sql("SELECT * FROM statewide_tax ORDER BY year", conn)
print(f"Rows in table: {len(check)}")
print(check.tail(3))  # confirm 2025 shows total_tax_millions = 6815.142

conn.close()

Rows in table: 28
    year  residential_millions  residential_pct  ci_real_pp_millions  \
25  2023              3491.817            56.15             1440.598   
26  2024              3696.259            57.07             1544.303   
27  2025              3958.152            58.08             1606.355   

    ci_real_pp_pct  utilities_millions  utilities_pct  ag_land_millions  \
25           23.17             659.147          10.60           352.887   
26           23.85             676.543          10.45           307.396   
27           23.57             722.627          10.60           281.232   

    ag_land_pct  oil_gas_millions  oil_gas_pct  all_other_millions  \
25         5.67           126.651         2.04             147.712   
26         4.75            96.791         1.49             155.045   
27         4.13            85.630         1.26             161.145   

    all_other_pct  total_tax_millions  
25           2.38            6218.812  
26           2.39            64

## Early observation

Residential property's share of total statewide tax has climbed from ~41% in
1998 to ~58% by 2025 — even independent of overall tax growth, homeowners are
absorbing a steadily larger share of the total property tax burden. This is
worth revisiting once the Reno County-specific and neighbor-state data are in
place, to see whether the same shift shows up locally.

**Output:** `statewide_tax` table in `kansas_tax.db`, plus
`data/processed/statewide_tax.csv` for portfolio/backup consistency with the
existing processed data.

In [10]:
historic_tax_df.to_csv("../data/processed/statewide_tax.csv", index=False)
print("Saved to data/processed/statewide_tax.csv")

Saved to data/processed/statewide_tax.csv


## Appraised Value (1998–2025)

Source: `PVDHistoricApprais.pdf` (KDOR). Raw market value before assessment
ratios are applied. Source PDF starts in 1992, but filtered to 1998+ to match
`statewide_valuation` and `statewide_tax`.

In [11]:
with pdfplumber.open("../data/raw/kdor/PVDHistoricApprais.pdf") as pdf:
    page = pdf.pages[0]
    apprais_raw_text = page.extract_text()

records = []

for line in apprais_raw_text.split("\n"):
    line = line.strip()
    match = re.match(r'^(\d{2})\s+(.*)$', line)
    if not match:
        continue

    year_2digit = match.group(1)
    rest = match.group(2)

    if '$' not in rest:
        continue

    parts = [p.strip() for p in rest.split('$') if p.strip()]

    if len(parts) != 7:
        continue  # skips the malformed 1992 row automatically

    yy = int(year_2digit)
    year = 1900 + yy if yy >= 90 else 2000 + yy

    if year < 1998:
        continue  # match statewide_valuation / statewide_tax range

    record = {'year': year}

    for i, cat in enumerate(categories):
        amt_str, pct_str = parts[i].split()
        record[f'{cat}_billions'] = float(amt_str.replace(',', ''))
        record[f'{cat}_pct'] = float(pct_str)

    record['total_appraised_billions'] = float(parts[6].replace(',', ''))

    records.append(record)

historic_apprais_df = pd.DataFrame(records)
print(f"Rows parsed: {len(historic_apprais_df)}")
historic_apprais_df

Rows parsed: 28


,year,residential_billions,residential_pct,ci_real_pp_billions,ci_real_pp_pct,utilities_billions,utilities_pct,ag_land_billions,ag_land_pct,oil_gas_billions,oil_gas_pct,all_other_billions,all_other_pct,total_appraised_billions
0,1998,64.043,59.97,20.908,19.58,9.236,8.65,4.429,4.15,4.920,4.61,3.253,3.05,106.790
1,1999,69.342,61.40,22.853,20.23,9.545,8.45,4.505,3.99,3.337,2.96,3.344,2.96,112.926
2,2000,76.227,62.54,24.511,20.11,9.436,7.74,4.775,3.92,3.184,2.61,3.752,3.08,121.886
3,2001,82.500,62.79,25.607,19.49,9.513,7.24,5.178,3.94,4.646,3.54,3.954,3.01,131.398
4,2002,87.755,64.12,26.297,19.22,9.285,6.78,5.356,3.91,4.121,3.01,4.056,2.96,136.870
5,2003,94.098,65.33,27.390,19.01,9.570,6.64,5.210,3.62,3.658,2.54,4.119,2.86,144.045
6,2004,99.709,65.40,28.174,18.48,10.074,6.61,5.355,3.51,5.022,3.29,4.133,2.71,152.467
7,2005,106.146,65.47,29.619,18.27,10.230,6.31,5.312,3.28,6.452,3.98,4.365,2.69,162.123
8,2006,113.762,65.40,31.704,18.10,10.265,5.90,5.129,2.90,8.422,4.84,4.654,2.68,173.936
9,2007,121.369,66.72,33.681,18.51,9.527,5.24,4.735,2.60,7.705,4.24,4.905,2.70,181.921


## Validation

28 rows (1998–2025). Checked 1998 ($106.790B) and 2025 ($359.792B) totals
against source PDF.

In [14]:
conn = sqlite3.connect("../data/processed/kansas_tax.db")

In [15]:
historic_apprais_df.to_sql("statewide_appraised", conn, if_exists="replace", index=False)
historic_apprais_df.to_csv("../data/processed/statewide_appraised.csv", index=False)

check = pd.read_sql("SELECT * FROM statewide_appraised ORDER BY year", conn)
print(f"Rows in table: {len(check)}")
print(check.tail(3))

Rows in table: 28
    year  residential_billions  residential_pct  ci_real_pp_billions  \
25  2023               236.487            74.56               44.776   
26  2024               253.636            75.78               48.479   
27  2025               271.427            75.44               50.523   

    ci_real_pp_pct  utilities_billions  utilities_pct  ag_land_billions  \
25           14.12              18.077           5.70             9.163   
26           14.48              16.239           4.85             8.088   
27           14.04              19.939           5.54             9.512   

    ag_land_pct  oil_gas_billions  oil_gas_pct  all_other_billions  \
25         2.89             3.334         1.05               5.338   
26         2.42             2.338         0.70               5.914   
27         2.64             2.174         0.60               6.217   

    all_other_pct  total_appraised_billions  
25           1.68                   317.175  
26           1.77  

In [16]:
comparison = pd.read_sql("""
    SELECT a.year,
           a.total_appraised_billions,
           v.total_assessed_value_billions,
           ROUND(v.total_assessed_value_billions / a.total_appraised_billions * 100, 2) AS assessed_pct_of_appraised
    FROM statewide_appraised a
    JOIN statewide_valuation v ON a.year = v.year
    ORDER BY a.year
""", conn)

print(comparison)

    year  total_appraised_billions  total_assessed_value_billions  \
0   1998                   106.790                         18.849   
1   1999                   112.926                         19.608   
2   2000                   121.886                         20.875   
3   2001                   131.398                         22.459   
4   2002                   136.870                         23.035   
5   2003                   144.045                         23.960   
6   2004                   152.467                         25.398   
7   2005                   162.123                         27.019   
8   2006                   173.936                         28.964   
9   2007                   181.921                         30.087   
10  2008                   187.802                         31.000   
11  2009                   185.470                         30.312   
12  2010                   182.145                         29.450   
13  2011                   184.308